In [1]:
! which python
! wget https://files.rcsb.org/download/5AWL.pdb

~/Tools/Spring2026_MLForChemicalSystems/.pixi/envs/default/bin/python
--2026-03-12 11:46:36--  https://files.rcsb.org/download/5AWL.pdb
Resolving files.rcsb.org (files.rcsb.org)... 99.84.118.112, 99.84.118.104, 99.84.118.21, ...
Connecting to files.rcsb.org (files.rcsb.org)|99.84.118.112|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘5AWL.pdb.7’

5AWL.pdb.7              [ <=>                ]  37.57K  --.-KB/s    in 0.03s   

2026-03-12 11:46:36 (1.14 MB/s) - ‘5AWL.pdb.7’ saved [38475]



In [2]:
import os
import glob
from typing import Tuple
import itertools
from copy import deepcopy
import math
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
plt.rcParams["font.family"] = 'Arial'
plt.rcParams.update({'font.size': 24})
plt.rcParams['svg.fonttype'] = 'none'

import rdkit
from rdkit import Chem
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import MDAnalysis as mda
from Bio.PDB.PDBParser import PDBParser

from vampnet_utils import *

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available. Using GPU.")
else:
    device = torch.device("cpu")
    print("GPU not available. Using CPU.")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

/home/sabari/Tools/Spring2026_MLForChemicalSystems/.pixi/envs/default/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU is available. Using GPU.


#### Parsing PDBs to get atom positions

Biopython is a comonly used library for parsing pdbs/doing bioinformatics-y things.
Here's how to use it to parse a pdb:

In [3]:
from Bio.PDB.PDBParser import PDBParser
parser = PDBParser(PERMISSIVE=1)
structure_id = "5AWL"
filename = "5AWL.pdb"
structure = parser.get_structure(structure_id, filename)


def get_CA_coords(structure: 'Bio.PDB.Structure.Structure') -> np.array:
    for model in structure.get_list():
        for chain in model.get_list():
            for residue in chain.get_list():
                if residue.has_id("CA"):
                    return residue["CA"].get_coord()
                    
def get_atom_coords(structure: 'Bio.PDB.Structure.Structure') -> np.array:
    at_coords = []
    for model in structure:
        for chain in model:
            for residue in chain.get_list():
                residue_id = residue.get_id()
                hetfield = residue_id[0]
                if hetfield[0] != "W":
                    for atom in residue:
                        at_coords.append(atom.get_coord())
    return np.array(at_coords)

get_atom_coords(structure).shape

(93, 3)

#### Multiframe PDBs
Since our file is a multiframe pdb, lets use MDAnalysis instead

In [4]:
u = mda.Universe('vampnet_training_data.pdb')
prot_atoms = u.select_atoms("not resname SOL and not resname WAT and not resname HOH")
coords = []

for ts in u.trajectory:
    # Get positions of all atoms in the current frame as a NumPy array
    # ts.positions also works and points to the same data
    coords.append(prot_atoms.positions)

    # You can also perform analysis here for the current frame
    # print(f"Frame {ts.frame}: {u.atoms.positions.mean()}")

# Convert the list of arrays into a single 3D NumPy array (frames, atoms, xyz)
coords = np.array(coords)
print(coords.shape)

(600, 22, 3)


In [5]:
pse = Chem.GetPeriodicTable()
{at:pse.GetAtomicWeight(at) for at in prot_atoms.elements}

{'H': 1.008, 'C': 12.011, 'O': 15.999, 'N': 14.007}

#### Custom VAMPNet dataset:

In [6]:
class VAMPDataset(Dataset):
    def __init__(self,
                 multi_pdb_path: str|os.PathLike):
        # Get atom coords
        self.at_symbols, self.at_coords = self._get_atom_coords(multi_pdb_path)
        # Get atomic weights to subtract out center of mass
        pse = Chem.GetPeriodicTable()
        self.at_weights = [pse.GetAtomicWeight(at) for at in self.at_symbols]
        # Subtract out center of mass using com_center function (definied in vampnet_utils)
        #self.at_coords_com = torch.tensor(com_center(self.at_coords, self.at_weights))
        #self.at_coords_com = self.at_coords_com.reshape(self.at_coords.shape[0],-1).float()
        # Make the (time, time + tau) pairs to train the model
        self.at_coords = torch.tensor(self.at_coords).reshape(self.at_coords.shape[0], -1).float()
        self.coords_combs = list(itertools.combinations(self.at_coords, 2))
        #self.coords_combs = list(itertools.combinations(self.at_coords_com, 2))
    
    def __len__(self):
        return len(self.coords_combs)

    def __getitem__(self, idx: int):
        return self.coords_combs[idx]
    
    def _get_atom_coords(self, path: str|os.PathLike) -> Tuple[np.array, torch.Tensor]:
        u = mda.Universe(path)
        prot_atoms = u.select_atoms("not resname SOL and not resname WAT and not resname HOH and not name H*")
        coords = []
        for ts in u.trajectory:
            coords.append(prot_atoms.positions)
        return prot_atoms.elements, np.array(coords)
    

#### Set up Dataloaders: 

In [7]:
BATCH_SIZE = 128
vamp_dataset = VAMPDataset('./vampnet_training_data.pdb')
print(f'Vampnet datset output: {next(iter(vamp_dataset))}')
# Check if any tensors are NaN
full_dataset = torch.stack([torch.stack((x, y)) for x, y in vamp_dataset.coords_combs]).reshape([-1, 30])
assert not torch.any(torch.isnan(full_dataset))

train_len = int(0.8*len(vamp_dataset))
valid_len = int(0.1*len(vamp_dataset))
test_len = len(vamp_dataset) - train_len - valid_len
generator = torch.Generator().manual_seed(42)

train_data, val_data, test_data = random_split(vamp_dataset,[train_len, valid_len, test_len], generator = generator)
train_loader = DataLoader(train_data, batch_size = BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size = BATCH_SIZE, shuffle=False)
test_loader = DataLoader(train_data, batch_size = BATCH_SIZE, shuffle=False)

Vampnet datset output: (tensor([12.0810, -1.4960, 18.4880, 13.2880, -0.8270, 18.0480, 14.2900, -1.3800,
        17.7420, 13.1930,  0.4770, 18.0150, 14.2860,  1.4350, 17.9860, 13.7570,
         2.6160, 17.1570, 14.9850,  1.9670, 19.2700, 16.0970,  2.5120, 19.2030,
        14.4280,  1.6780, 20.4690, 15.1230,  2.0770, 21.6800]), tensor([15.8140, 28.2880, 15.1560, 16.4710, 29.5450, 14.6370, 17.1140, 30.1650,
        15.5050, 16.5050, 29.8300, 13.3960, 17.2370, 30.9450, 12.7460, 17.2590,
        30.7770, 11.2210, 16.7500, 32.3360, 13.1310, 17.5340, 33.2450, 13.3480,
        15.4140, 32.4180, 13.2930, 14.6450, 33.5420, 13.6840]))


In [8]:
class VampNetModel(nn.Module):
    """
    Vampnet model class; consists of two MLPs, one for each time point
    """
    def __init__(self,
                n_layers: int = 6,
                input_size: int = 30,
                n_states: int = 6,
                dropout_prob: float = 0.1):
        super().__init__()
        # Calculate layer dimensions using eqn 16.
        red_ratio = (input_size/n_states) ** (1/n_layers)
        layer_dims = [30]
        i = 0
        while i < n_layers-1:
            layer_dims.append(math.ceil(layer_dims[-1]/red_ratio))
            i += 1
        layer_dims[-1] = n_states 
        # Layer defns.
        t_head = []
        for i in range(len(layer_dims)-1):
            t_head.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            t_head.append(nn.Dropout(p = dropout_prob))
            t_head.append(nn.SiLU())
        t_head.pop()
        t_head.pop()
        t_head.append(nn.Softmax())
        self.t_head = nn.Sequential(*t_head)
        self.tau_head = deepcopy(self.t_head)

    def forward(self, x):
        t_in, tau_in = x
        t_pred = self.t_head(t_in)
        tau_pred = self.tau_head(tau_in)
        return t_pred, tau_pred

In [9]:
model = VampNetModel()
print(model)
loss_fn = VampNetLoss()
LEARNING_RATE = 0.001
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

def train_one_epoch():
    running_loss = 0.
    last_loss = 0.
    for i, inputs in enumerate(train_loader):
        optimizer.zero_grad()
        t, tau = inputs
        t_pred, tau_pred = model(inputs)
        loss = loss_fn(t_pred, tau_pred)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss/len(train_loader)

VampNetModel(
  (t_head): Sequential(
    (0): Linear(in_features=30, out_features=23, bias=True)
    (1): Dropout(p=0.1, inplace=False)
    (2): SiLU()
    (3): Linear(in_features=23, out_features=18, bias=True)
    (4): Dropout(p=0.1, inplace=False)
    (5): SiLU()
    (6): Linear(in_features=18, out_features=14, bias=True)
    (7): Dropout(p=0.1, inplace=False)
    (8): SiLU()
    (9): Linear(in_features=14, out_features=11, bias=True)
    (10): Dropout(p=0.1, inplace=False)
    (11): SiLU()
    (12): Linear(in_features=11, out_features=6, bias=True)
    (13): Softmax(dim=None)
  )
  (tau_head): Sequential(
    (0): Linear(in_features=30, out_features=23, bias=True)
    (1): Dropout(p=0.1, inplace=False)
    (2): SiLU()
    (3): Linear(in_features=23, out_features=18, bias=True)
    (4): Dropout(p=0.1, inplace=False)
    (5): SiLU()
    (6): Linear(in_features=18, out_features=14, bias=True)
    (7): Dropout(p=0.1, inplace=False)
    (8): SiLU()
    (9): Linear(in_features=14, out_f

/home/sabari/Tools/Spring2026_MLForChemicalSystems/.pixi/envs/default/lib/python3.13/site-packages/sympy/external/gmpy.py:139: UserWarning: gmpy2 version is too old to use (2.0.0 or newer required)
  gmpy = import_module('gmpy2', min_module_version=_GMPY2_MIN_VERSION,


In [10]:
model(next(iter(train_data)))

/home/sabari/Tools/Spring2026_MLForChemicalSystems/.pixi/envs/default/lib/python3.13/site-packages/torch/nn/modules/module.py:1776: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


(tensor([0.1459, 0.1811, 0.1656, 0.1540, 0.2204, 0.1330],
        grad_fn=<SoftmaxBackward0>),
 tensor([0.1460, 0.1869, 0.1682, 0.1467, 0.2151, 0.1371],
        grad_fn=<SoftmaxBackward0>))

In [12]:
EPOCHS = 20

epoch_number = 0
best_vloss = 1_000_000.

timestamp = datetime.datetime.now().strftime("%Y%m%d")    

for epoch in range(EPOCHS):
    print('EPOCH {}:'.format(epoch_number + 1))
    model.train(True)
    avg_loss = train_one_epoch()
    running_vloss = 0.0
    model.eval()
    with torch.no_grad():
        for i, v_inputs in enumerate(val_loader):
            v_t_pred, v_tau_pred = model(v_inputs)
            v_loss = loss_fn(v_t_pred, v_tau_pred)
            running_vloss += v_loss

    avg_vloss = running_vloss / (i + 1)
    print('LOSS train {} valid {}'.format(avg_loss, avg_vloss))

    # Track best performance, and save the model's state
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        model_path = './training_results/model_{}_{}'.format(timestamp, epoch_number)
        torch.save(model.state_dict(), model_path)
    epoch_number += 1

EPOCH 1:
LOSS train -8958634537171802.0 valid -1946631918321664.0
EPOCH 2:
LOSS train -5780372800528748.0 valid -426773854552064.0
EPOCH 3:
LOSS train -2328307199521330.0 valid -2490823333642240.0
EPOCH 4:
LOSS train -1114924301645980.2 valid -1.1600466148327424e+16
EPOCH 5:
LOSS train -2165414002639837.5 valid -1.780237637517312e+16
EPOCH 6:
LOSS train -3781673588222332.5 valid -65983188303872.0
EPOCH 7:
LOSS train -4661423900300648.0 valid -1824052612169728.0
EPOCH 8:
LOSS train -3260485229539952.0 valid -2333595586789376.0
EPOCH 9:
LOSS train -3048508469138427.0 valid -2969295575318528.0
EPOCH 10:
LOSS train -1877799885365358.5 valid -4303980821741568.0
EPOCH 11:
LOSS train -3853995627726114.5 valid -3.634129024948634e+16
EPOCH 12:
LOSS train -2.5744710120647996e+16 valid -2352897169817600.0
EPOCH 13:
LOSS train -2262662268304887.5 valid -386567726170112.0
EPOCH 14:
LOSS train -2328029856945171.0 valid -489119096504320.0
EPOCH 15:
LOSS train -4131563627998373.0 valid -55079583350784